::::{margin}
:::{card} Spin-1 daughter in a three-body Dalitz model
TR-037
^^^
Model $\overline{B}^0\to D^{*+}\pi^-\pi^0$ twice: as a three-body decay with a stable spin-1 $D^{*+}$ in AmpForm-DPD, and as the four-body decay $\overline{B}^0\to D^0\pi^+\pi^-\pi^0$ in AmpForm with the $D^0\pi^+$ subsystem pinned to the $D^{*+}$.
:::
::::

<!-- cspell:ignore cividis Clebsch Dalitz dstar Gordan isfinite isobar Kibble linalg naturality pointwise propto qrules subsystem subsystems varphi -->

# $\overline{B}^0\to D^{*+}\pi^-\pi^0$ with a spin-1 daughter

Every isobar model in this collection so far has spinless final-state particles, so the only spin in the problem sits on the resonances. $\overline{B}^0\to D^{*+}\pi^-\pi^0$ breaks that pattern: the $D^{*+}$ is a $J^P=1^-$ particle that itself decays. This report asks what changes when one of the three "final-state" particles carries spin, and answers the question twice over.

The first model treats the $D^{*+}$ as a stable spin-1 particle and formulates a genuine three-body decay with [AmpForm-DPD](https://ampform-dpd.readthedocs.io/stable/). The second reconstructs the $D^{*+}$ through $D^{*+}\to D^0\pi^+$, so that the process becomes the four-body decay $\overline{B}^0\to D^0\pi^+\pi^-\pi^0$, and formulates it with [AmpForm](https://ampform.readthedocs.io/stable/). Both start from decay chains generated by [QRules](https://qrules.readthedocs.io/stable/) and are evaluated with [TensorWaves](https://tensorwaves.readthedocs.io/).

The channel is a natural Belle II target and is poorly measured. The Review of Particle Physics {cite}`ParticleDataGroup:2024cfk` quotes $\mathcal{B}(\overline{B}^0\to D^{*+}\pi^-\pi^0)=(1.5\pm0.5)\%$ from a single 51-event ARGUS measurement, and the quasi-two-body rate $\mathcal{B}(\overline{B}^0\to D^{*+}\rho^-)=(2.2^{+1.8}_{-2.7})\times10^{-3}$ carries a scale factor $S=5.2$. No amplitude analysis of this final state has been published. The neighboring $D^{(*)}\pi^+\pi^-$ modes have been analyzed by Belle {cite}`Belle:2006wbx`, BaBar {cite}`BaBar:2010rll` and LHCb {cite}`LHCb:2015klp`, and the $D^{*+}\pi^-$ subsystem that appears here is the one LHCb used to determine the quantum numbers of excited charm mesons in $B^-\to D^{*+}\pi^-\pi^-$ {cite}`LHCb:2019juy`. What the $\pi^0$ adds is access to the *charged* partners $D_1(2420)^+$ and $D_2^*(2460)^+$ through the $D^{*+}\pi^0$ subsystem.

The couplings used below are illustrative. Their helicity structure follows from parity and angular momentum, but their magnitudes and phases are chosen so that all three subsystems are visible. They are neither a fit to data nor a prediction.

In [ ]:
import logging
import os
import warnings
from importlib.metadata import version

import ampform
import attrs
import matplotlib.pyplot as plt
import numpy as np
import phasespace
import qrules
import sympy as sp
from ampform.dynamics.builder import create_relativistic_breit_wigner_with_ff
from ampform.dynamics.form_factor import FormFactor
from ampform.io import aslatex
from ampform_dpd import DalitzPlotDecompositionBuilder
from ampform_dpd.adapter.qrules import normalize_state_ids, to_three_body_decay
from ampform_dpd.dynamics.builder import formulate_breit_wigner_with_form_factor
from ampform_dpd.io import as_markdown_table
from IPython.display import Markdown, Math
from matplotlib.colors import LogNorm
from matplotlib_inline.backend_inline import set_matplotlib_formats
from qrules.particle import Parity
from qrules.transition import InteractionType, ReactionInfo, StateTransitionManager
from sympy.physics.quantum.cg import CG
from tensorwaves.data.transform import SympyDataTransformer
from tensorwaves.function.sympy import create_parametrized_function

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"


set_matplotlib_formats("svg")
logging.getLogger("qrules").setLevel(logging.ERROR)
for package in ("qrules", "ampform", "ampform-dpd", "tensorwaves", "phasespace"):
    print(f"{package}: {version(package)}")

## Is the $D^{*+}$ narrow enough?

Treating a resonance as a stable final-state particle is the narrow-width approximation, and $\Gamma/m$ controls it. For the $D^{*+}$ that ratio is three to four orders of magnitude smaller than for anything else in this decay, so the approximation is not in question.

In [ ]:
widths = {
    R"D^{*}(2010)^{+}": (2.01027, 83.4e-6),
    "D_1(2420)": (2.4221, 31.3e-3),
    R"D_2^{*}(2460)": (2.4611, 47.3e-3),
    R"\rho(770)": (0.77511, 149.1e-3),
}
rows = [
    R"| resonance | $m$ [GeV] | $\Gamma$ [GeV] | $\Gamma/m$ |",
    "|---|---:|---:|---:|",
]
rows += [
    f"| ${name}$ | {mass:.5g} | {width:.4g} | {width / mass:.1e} |"
    for name, (mass, width) in widths.items()
]
Markdown("\n".join(rows))

The $D^{*+}$ line width is $\Gamma=83.4\pm1.8\;\mathrm{keV}$ {cite}`BaBar:2013zgp`, so the $D^{*+}$ can be held at its pole mass throughout. What the three-body reduction costs is therefore not lineshape accuracy but information: the $D^{*+}$ carries spin, and summing over its helicity discards angular correlations that its own decay would reveal. The last two sections make that statement quantitative.

## Decay chains for the three-body model

Label $\overline{B}^0$ by 0 and $(D^{*+},\pi^-,\pi^0)$ by $(1,2,3)$, so that the Dalitz-plot decomposition variables are

$$\sigma_1=m^2(\pi^-\pi^0),\qquad \sigma_2=m^2(D^{*+}\pi^0),\qquad \sigma_3=m^2(D^{*+}\pi^-).$$

Each subsystem carries its own resonances: $\rho(770)^-$ in $\sigma_1$, the charged $D^{**+}$ states in $\sigma_2$, and the neutral $D^{**0}$ states in $\sigma_3$. We keep the two narrow $P$-wave charm states, $D_1(2420)$ and $D_2^*(2460)$, in both charge configurations. The broad $D_1(2430)$ is left out because QRules only carries its neutral charge state, which would break the isospin symmetry of the model. Unlike TR-036, there are no identical particles here, so no Bose symmetrization is needed.

Two properties of the QRules particle database need attention first. The charged $D_1(2420)^\pm$ carry no parity, which AmpForm-DPD requires, so we copy the $J^P=1^+$ of their neutral partners. And because the $\overline{B}^0$ decay is weak, QRules has to be allowed to violate parity; left at that, it also violates parity at the *strong* resonance vertices and offers unphysical decay waves. We therefore select each resonance's decay wave explicitly: $P$ wave for $\rho(770)^-\to\pi^-\pi^0$, and $D$ wave for $D^{**}\to D^{*+}\pi$, the wave that heavy-quark spin symmetry singles out for the narrow $j_q=3/2$ doublet.

In [ ]:
DECAY_WAVE = {
    "rho(770)-": 1,
    "D(1)(2420)0": 2,
    "D(1)(2420)+": 2,
    "D(2)*(2460)0": 2,
    "D(2)*(2460)+": 2,
}


def load_particles():
    particle_db = qrules.particle.load_pdg()
    for name in ("D(1)(2420)+", "D(1)(2420)-"):
        particle = particle_db[name]
        assert particle.parity is None
        particle_db.remove(name)
        particle_db.add(attrs.evolve(particle, parity=Parity(+1)))
    return particle_db


def has_physical_decay_wave(transition, node_id):
    (resonance,) = transition.intermediate_states.values()
    return (
        transition.interactions[node_id].l_magnitude
        == DECAY_WAVE[resonance.particle.name]
    )


reaction = qrules.generate_transitions(
    initial_state="B~0",
    final_state=["D*(2010)+", "pi-", "pi0"],
    allowed_intermediate_particles=list(DECAY_WAVE),
    allowed_interaction_types=["strong", "weak"],
    formalism="canonical-helicity",
    particle_db=load_particles(),
    number_of_threads=1,
)
reaction = ReactionInfo(
    [t for t in reaction.transitions if has_physical_decay_wave(t, node_id=1)],
    reaction.formalism,
)
decay = to_three_body_decay(normalize_state_ids(reaction).transitions, min_ls=True)
assert [decay.states[i].name for i in range(4)] == ["B~0", "D*(2010)+", "pi-", "pi0"]
assert {chain.resonance.name for chain in decay.chains} == set(DECAY_WAVE)
resonances = {chain.resonance.name: chain.resonance for chain in decay.chains}
subsystem_ids = {chain.resonance.name: chain.spectator.index for chain in decay.chains}
Markdown(as_markdown_table(decay))

## Three-body amplitude with AmpForm-DPD

The Dalitz-plot decomposition {cite}`JPAC:2019ufm` writes the amplitude as a sum over subsystems in which each chain is rotated into a common frame. With spinless daughters those alignment rotations are trivial, which is why they never appeared in TR-036. Here the $D^{*+}$ carries helicity indices through them, and the Wigner rotation angles $\zeta^1_{k(1)}$ below are exactly the price of that.

The builder is given the chains and one dynamics builder per resonance. As in TR-036, AmpForm-DPD v0.2.4 passes $\sigma_k^2$ to the decay form factor and the pole mass $m_R$ as the production daughter mass; the correction below replaces those by $\sigma_k$ and $\sqrt{\sigma_k}$ of the resonance's *own* subsystem.

In [ ]:
sigma1, sigma2, sigma3 = sigmas = sp.symbols("sigma1:4", nonnegative=True)
m0, m1, m2, m3 = sp.symbols("m0:4", nonnegative=True)

builder = DalitzPlotDecompositionBuilder(decay)
for chain in decay.chains:
    builder.dynamics_choices.register_builder(
        chain.resonance.name, formulate_breit_wigner_with_form_factor
    )
model = builder.formulate(
    reference_subsystem=1, cleanup_summations=True, use_coefficients=True
)

running_mass = {
    f"m_{{{chain.resonance.latex}}}": sigmas[chain.spectator.index - 1]
    for chain in decay.chains
}
form_factor_corrections = {}
for expression in model.amplitudes.values():
    for ff in expression.atoms(FormFactor):
        if ff.s in {s**2 for s in sigmas}:
            form_factor_corrections[ff] = FormFactor(
                sp.sqrt(ff.s), ff.m1, ff.m2, ff.angular_momentum, ff.meson_radius
            )
        elif ff.s == m0**2 and str(ff.m1) in running_mass:
            form_factor_corrections[ff] = FormFactor(
                m0**2,
                sp.sqrt(running_mass[str(ff.m1)]),
                ff.m2,
                ff.angular_momentum,
                ff.meson_radius,
            )
model.amplitudes.update({
    symbol: expression.xreplace(form_factor_corrections)
    for symbol, expression in model.amplitudes.items()
})
for expression in model.amplitudes.values():
    for ff in expression.atoms(FormFactor):
        assert ff.s in set(sigmas) | {m0**2}
        if ff.s == m0**2:
            assert isinstance(ff.m1, sp.Pow)
Math(aslatex(model.amplitudes, terms_per_line=1))

The nine amplitude components are labelled by the $\overline{B}^0$ helicity, the $D^{*+}$ helicity and the two pion helicities; only the $D^{*+}$ index is non-trivial, so there are three components per subsystem. The alignment angles and dynamics layers are rendered below.

In [ ]:
x, y, z = sp.symbols("x y z", real=True)
kinematics = {
    symbol: expression
    for symbol, expression in model.variables.items()
    if str(symbol).startswith(("theta", R"\zeta"))
}
Math(aslatex(kinematics))

## Helicity couplings of a spin-1 daughter

With a spin-1 daughter, one complex coupling per decay chain is no longer enough: each chain comes with a coupling $\mathcal{H}^R_\lambda$ for every $D^{*+}$ helicity $\lambda\in\{-1,0,+1\}$, fifteen in total. They are not independent. The $\overline{B}^0$ vertex is weak, so parity constrains nothing there, but the resonance decays are strong, and the single decay wave selected above fixes the helicity pattern up to one overall coupling per resonance.

For $R\to D^{*+}\pi$ in a $D$ wave, the pattern is the Clebsch-Gordan coefficient $\langle L\,0;S\,\lambda\mid J_R\,\lambda\rangle$ with $L=2$ and $S=1$:

$$\mathcal{H}^R_\lambda \propto \langle 2\,0;1\,\lambda\mid J_R\,\lambda\rangle .$$

For $J_R=1$ this is even in $\lambda$; for $J_R=2$ it is odd, and $\langle2\,0;1\,0\mid2\,0\rangle=0$ makes the $D_2^*(2460)$ longitudinal coupling vanish outright. The same conclusion follows from the parity relation $\mathcal{H}_{-\lambda}=P_R(-1)^{J_R-1}\mathcal{H}_{\lambda}$. For $\overline{B}^0\to D^{*+}\rho^-$, on the other hand, all three couplings are free: this is the vector-vector configuration whose longitudinal fraction $f_L$ is the quantity experiments quote. We take $f_L=0.885$ and split the transverse strength equally.

That leaves seven free complex numbers, one per resonance plus two extra for the $\rho^-$ polarization, which is exactly the number of independent $LS$ couplings in the canonical basis.

In [ ]:
HELICITY_SLOT = {1: 1, 2: 3, 3: 2}
LONGITUDINAL_FRACTION = 0.885


def coupling_symbol(resonance_name, helicity):
    resonance = resonances[resonance_name]
    subsystem = subsystem_ids[resonance_name]
    indices = [0, 0, 0, 0]
    indices[HELICITY_SLOT[subsystem]] = helicity
    if subsystem == 1:
        indices[0] = helicity
    base = sp.IndexedBase(Rf"\mathcal{{H}}^\mathrm{{{resonance.latex}}}")
    return base[tuple(indices)]


def decay_wave_pattern(resonance_name):
    spin = resonances[resonance_name].spin
    wave = DECAY_WAVE[resonance_name]
    return {
        helicity: float(CG(wave, 0, 1, helicity, spin, helicity).doit())
        for helicity in (-1, 0, 1)
    }


transverse = np.sqrt((1 / LONGITUDINAL_FRACTION - 1) / 2)
patterns = {
    name: {-1: transverse, 0: 1.0, +1: transverse}
    if subsystem_ids[name] == 1
    else decay_wave_pattern(name)
    for name in DECAY_WAVE
}
assert patterns["D(2)*(2460)0"][0] == 0
assert patterns["D(1)(2420)0"][-1] == patterns["D(1)(2420)0"][+1]
rows = [
    R"| resonance | $\lambda=-1$ | $\lambda=0$ | $\lambda=+1$ |",
    "|---|---:|---:|---:|",
]
rows += [
    f"| ${resonances[name].latex}$ | "
    + " | ".join(f"{pattern[h]:+.4f}" for h in (-1, 0, 1))
    + " |"
    for name, pattern in patterns.items()
]
Markdown("\n".join(rows))

The couplings themselves are set from illustrative component fractions. As in TR-036 the fractions are defined through the diagonal integrals $N_R=\int_\mathcal{D}I_R\,\mathrm{d}\sigma_3\,\mathrm{d}\sigma_1$ of each component evaluated on its own, so that the numbers mean the same thing for every lineshape. Three-body phase space is flat in $\mathrm{d}\sigma_3\,\mathrm{d}\sigma_1$ for a scalar parent, so no extra momentum weight is needed.

In [ ]:
expression = model.full_expression.xreplace(model.variables).doit()
expression = expression.xreplace(model.masses)
couplings = {
    symbol: value
    for symbol, value in model.parameter_defaults.items()
    if isinstance(symbol, sp.Indexed)
}
fixed_parameters = {
    symbol: value
    for symbol, value in model.parameter_defaults.items()
    if symbol not in couplings
}
expression = expression.xreplace(fixed_parameters)
assert set(couplings) == {coupling_symbol(n, h) for n in DECAY_WAVE for h in (-1, 0, 1)}
intensity_function = create_parametrized_function(expression, couplings, backend="jax")

MASSES = [float(model.masses[sp.Symbol(f"m{i}", nonnegative=True)]) for i in range(4)]
SIGMA_SUM = sum(mass**2 for mass in MASSES)


def set_couplings(scales):
    parameters = {}
    for name, pattern in patterns.items():
        for helicity, weight in pattern.items():
            symbol = str(coupling_symbol(name, helicity))
            parameters[symbol] = complex(scales.get(name, 0)) * weight
    intensity_function.update_parameters(parameters)


def evaluate(scales, data):
    set_couplings(scales)
    return np.asarray(intensity_function(data))

## Physical region and the Dalitz plot

The physical region follows from the Kibble function $\phi<0$ {cite}`Byckling:1971vca`, evaluated on a regular grid in $\left(\sigma_3,\sigma_1\right)$.

In [ ]:
def kallen(x, y, z):
    return x**2 + y**2 + z**2 - 2 * (x * y + y * z + z * x)


def is_inside(s1, s2, s3):
    parent = MASSES[0] ** 2
    return (
        kallen(
            kallen(s2, MASSES[2] ** 2, parent),
            kallen(s3, MASSES[3] ** 2, parent),
            kallen(s1, MASSES[1] ** 2, parent),
        )
        < 0
    )


def dalitz_grid(n_bins):
    parent, m_dstar, m_pim, m_pi0 = MASSES
    x_edges = np.linspace((m_dstar + m_pim) ** 2, (parent - m_pi0) ** 2, n_bins + 1)
    y_edges = np.linspace((m_pim + m_pi0) ** 2, (parent - m_dstar) ** 2, n_bins + 1)
    s3, s1 = np.meshgrid(
        (x_edges[1:] + x_edges[:-1]) / 2, (y_edges[1:] + y_edges[:-1]) / 2
    )
    s2 = SIGMA_SUM - s1 - s3
    inside = is_inside(s1, s2, s3)
    data = {"sigma1": s1[inside], "sigma2": s2[inside], "sigma3": s3[inside]}
    cell_area = (x_edges[1] - x_edges[0]) * (y_edges[1] - y_edges[0])
    return x_edges, y_edges, inside, data, cell_area


x_edges, y_edges, inside, grid_data, cell_area = dalitz_grid(500)
norms = {
    name: float(np.sum(evaluate({name: 1.0}, grid_data)) * cell_area)
    for name in DECAY_WAVE
}
assert all(np.isfinite(value) and value > 0 for value in norms.values())
*_, coarse_data, coarse_area = dalitz_grid(250)
coarse_norms = {
    name: float(np.sum(evaluate({name: 1.0}, coarse_data)) * coarse_area)
    for name in DECAY_WAVE
}
grid_stability = max(abs(coarse_norms[name] / norms[name] - 1) for name in norms)
assert grid_stability < 0.02, grid_stability

fractions = {
    "rho(770)-": 0.45,
    "D(2)*(2460)0": 0.20,
    "D(1)(2420)0": 0.15,
    "D(2)*(2460)+": 0.12,
    "D(1)(2420)+": 0.08,
}
phases = {
    "rho(770)-": 0.0,
    "D(2)*(2460)0": 120.0,
    "D(1)(2420)0": -60.0,
    "D(2)*(2460)+": 150.0,
    "D(1)(2420)+": -30.0,
}
scales = {
    name: np.sqrt(fraction / norms[name]) * np.exp(1j * np.deg2rad(phases[name]))
    for name, fraction in fractions.items()
}
density_values = evaluate(scales, grid_data)
coherent_integral = float(np.sum(density_values) * cell_area)
assert np.isfinite(density_values).all()
assert (density_values >= 0).all()
rows = [
    "| resonance | subsystem | input fraction [%] | phase [deg] | model fraction [%] |",
    "|---|---|---:|---:|---:|",
]
for name, fraction in fractions.items():
    model_fraction = abs(scales[name]) ** 2 * norms[name] / coherent_integral
    rows.append(
        f"| ${resonances[name].latex}$ | $\\sigma_{subsystem_ids[name]}$ "
        f"| {100 * fraction:g} | {phases[name]:g} | {100 * model_fraction:.2f} |"
    )
rows.append(
    f"\nLargest change in component integrals on grid refinement: {100 * grid_stability:.2f}%."
)
Markdown("\n".join(rows))

The model fractions differ from the input fractions only through interference, which is why they do not sum to 100%. The Dalitz plot below shows the coherent intensity relative to its maximum.

In [ ]:
density = np.full(inside.shape, np.nan)
density[inside] = density_values
relative_density = density / np.nanmax(density)

fig, ax = plt.subplots(figsize=(7.5, 5.6), layout="constrained")
mesh = ax.pcolormesh(
    x_edges,
    y_edges,
    np.ma.masked_invalid(relative_density),
    cmap="cividis",
    norm=LogNorm(vmin=1e-5, vmax=1),
    rasterized=True,
)
ax.set(
    xlabel=R"$\sigma_3 = m^2(D^{*+}\pi^-)$ [GeV$^2$]",
    ylabel=R"$\sigma_1 = m^2(\pi^-\pi^0)$ [GeV$^2$]",
    title=R"$\overline{B}^0\to D^{*+}\pi^-\pi^0$ - illustrative isobar model",
)
fig.colorbar(mesh, ax=ax, label=R"$I/I_\mathrm{max}$", extend="min")
fig.savefig("dalitz.svg")
plt.show()

The three subsystems are cleanly separated: the $D^{**0}$ states form the vertical band near $\sigma_3\approx6\;\mathrm{GeV}^2$, the $\rho(770)^-$ the horizontal band at $\sigma_1\approx0.6\;\mathrm{GeV}^2$, and the $D^{**+}$ states the diagonal band of constant $\sigma_2$.

## What the Dalitz plot cannot see

Because the $D^{*+}$ helicity is not observed, the three-body intensity is an incoherent sum over it, $I=\sum_\lambda\left|\sum_R\mathcal{A}^R_\lambda\right|^2$. Two things follow, and both are easy to check numerically.

The first is that components whose helicity patterns behave oppositely under $\lambda\to-\lambda$ do not interfere at all. Writing the cross term as $\sum_\lambda 2\,\mathrm{Re}\left(a_\lambda b_\lambda^*\right)X_\lambda$ with an even $X_\lambda$, an even $a$ against an odd $b$ gives a summand that is odd in $\lambda$ and cancels between $+\lambda$ and $-\lambda$. The check below switches on two components at a time and isolates $I_{ab}-I_a-I_b$ at random points inside the physical region: for equal $\lambda$-parity it is of order one, for opposite $\lambda$-parity it vanishes pointwise to machine precision, not merely on integration.

The second is that the relative sign of the two transverse $\rho^-$ couplings drops out entirely, because $\mathcal{H}_{+1}=+\mathcal{H}_{-1}$ and $\mathcal{H}_{+1}=-\mathcal{H}_{-1}$ differ only by that odd part. The longitudinal and transverse configurations themselves are *not* degenerate: they populate the Dalitz plane quite differently, so $f_L$ is in principle measurable from the Dalitz distribution alone.

In [ ]:
def sample_physical_region(n_points, seed=37):
    parent, m_dstar, m_pim, m_pi0 = MASSES
    rng = np.random.default_rng(seed)
    collected = {"sigma1": [], "sigma3": []}
    while sum(len(v) for v in collected["sigma1"]) < n_points:
        s3 = rng.uniform((m_dstar + m_pim) ** 2, (parent - m_pi0) ** 2, 20_000)
        s1 = rng.uniform((m_pim + m_pi0) ** 2, (parent - m_dstar) ** 2, 20_000)
        accepted = is_inside(s1, SIGMA_SUM - s1 - s3, s3)
        collected["sigma1"].append(s1[accepted])
        collected["sigma3"].append(s3[accepted])
    s1 = np.concatenate(collected["sigma1"])[:n_points]
    s3 = np.concatenate(collected["sigma3"])[:n_points]
    return {"sigma1": s1, "sigma2": SIGMA_SUM - s1 - s3, "sigma3": s3}


test_data = sample_physical_region(2_000)
helicity_parity = {
    name: "even" if pattern[+1] * pattern[-1] >= 0 else "odd"
    for name, pattern in patterns.items()
}
rows = [
    R"| pair | same $\lambda$-parity | $\left\langle\left|I_{ab}-I_a-I_b\right|\right\rangle/\sqrt{I_aI_b}$ |",
    "|---|:-:|---:|",
]
for i, first in enumerate(DECAY_WAVE):
    for second in list(DECAY_WAVE)[i + 1 :]:
        i_a = evaluate({first: 1.0}, test_data)
        i_b = evaluate({second: 1.0}, test_data)
        cross = evaluate({first: 1.0, second: 1.0}, test_data) - i_a - i_b
        typical = float(np.mean(np.abs(cross) / np.sqrt(i_a * i_b)))
        same = helicity_parity[first] == helicity_parity[second]
        assert typical > 1e-2 if same else typical < 1e-9, (first, second, typical)
        rows.append(
            f"| ${resonances[first].latex}$, ${resonances[second].latex}$ "
            f"| {'yes' if same else 'no'} | {typical:.1e} |"
        )

saved_pattern = patterns["rho(770)-"]
polarization_densities = {}
for label, pattern in (
    ("longitudinal", {-1: 0.0, 0: 1.0, +1: 0.0}),
    ("transverse, equal signs", {-1: 1.0, 0: 0.0, +1: +1.0}),
    ("transverse, opposite signs", {-1: 1.0, 0: 0.0, +1: -1.0}),
):
    patterns["rho(770)-"] = pattern
    values = evaluate({"rho(770)-": 1.0}, test_data)
    polarization_densities[label] = values / values.sum()
patterns["rho(770)-"] = saved_pattern


def relative_difference(first, second):
    return float(
        np.max(np.abs(polarization_densities[first] - polarization_densities[second]))
        / polarization_densities[first].max()
    )


transverse_sign = relative_difference(
    "transverse, equal signs", "transverse, opposite signs"
)
polarization = relative_difference("longitudinal", "transverse, equal signs")
assert transverse_sign == 0, transverse_sign
assert polarization > 0.5, polarization
rows.append(
    "\nRelative difference between the two transverse sign conventions: "
    f"{transverse_sign:.0e}. Between longitudinal and transverse: {100 * polarization:.0f}%."
)
Markdown("\n".join(rows))

So the Dalitz plot is not blind to the $\rho^-$ polarization as such, but it is blind to one number: the relative phase of the two transverse amplitudes. That number is what an angular analysis is for, and it needs the $D^{*+}$ decay.

## Four-body model with AmpForm

Reconstructing $D^{*+}\to D^0\pi^+$ turns the process into $\overline{B}^0\to D^0\pi^+\pi^-\pi^0$. Left alone, QRules would enumerate every way of pairing four particles, most of which have nothing to do with this decay. `StateTransitionManager.add_final_state_grouping` pins the topology: requiring $D^0$ and $\pi^+$ to originate from a common node keeps only the three chains that pass through a $D^0\pi^+$ isobar.

Pinning the topology is not quite the same as pinning the particle. `allowed_intermediate_particles` applies to *every* intermediate edge, so QRules still offers chains with a $D^{**}$ in the inner $D^0\pi^+$ slot. The filter below removes those and keeps the same physical decay waves as the three-body model.

No spin alignment amplitudes are needed here, unlike in the three-body model: all four final-state particles are spinless, so the Wigner rotations between decay chains act on nothing.

In [ ]:
FOUR_BODY_WAVE = {**DECAY_WAVE, "D*(2010)+": 1}


def keeps_pinned_chain(transition):
    names = {state.particle.name for state in transition.intermediate_states.values()}
    if "D*(2010)+" not in names or len(names) != 2:
        return False
    topology = transition.topology
    (initial_edge,) = topology.incoming_edge_ids
    production_node = topology.edges[initial_edge].ending_node_id
    for node_id in topology.nodes:
        if node_id == production_node:
            continue
        (parent,) = topology.get_edge_ids_ingoing_to_node(node_id)
        resonance = transition.states[parent].particle.name
        if transition.interactions[node_id].l_magnitude != FOUR_BODY_WAVE[resonance]:
            return False
    return True


stm = StateTransitionManager(
    initial_state=["B~0"],
    final_state=["D0", "pi+", "pi-", "pi0"],
    allowed_intermediate_particles=list(FOUR_BODY_WAVE),
    formalism="canonical-helicity",
    particle_db=load_particles(),
    max_angular_momentum=2,
    max_spin_magnitude=2,
    number_of_threads=1,
)
stm.add_final_state_grouping([["D0", "pi+"]])
stm.set_allowed_interaction_types([InteractionType.STRONG, InteractionType.WEAK])
four_body_reaction = stm.find_solutions(stm.create_problem_sets())
four_body_reaction = ReactionInfo(
    [t for t in four_body_reaction.transitions if keeps_pinned_chain(t)],
    formalism="helicity",
)
assert len({t.topology for t in four_body_reaction.transitions}) == 3
assert {s.name for s in four_body_reaction.final_state.values()} == {
    "D0",
    "pi+",
    "pi-",
    "pi0",
}
Markdown(
    "The pinned reaction has "
    f"{len(four_body_reaction.transitions)} transitions over three topologies, with intermediate states "
    + ", ".join(
        f"${four_body_reaction.get_intermediate_particles()[n].latex}$"
        for n in sorted(four_body_reaction.get_intermediate_particles().names)
    )
    + "."
)

The reaction is handed to AmpForm in the helicity basis with free helicity couplings, so that its parameters correspond one-to-one to the couplings of the three-body model: one per vertex and $D^{*+}$ helicity. The couplings that the canonical solutions forbid, such as the longitudinal $D_2^*(2460)\to D^{*+}\pi$ coupling, are simply absent.

In [ ]:
four_body_builder = ampform.get_builder(four_body_reaction)
four_body_builder.config.use_helicity_couplings = True
four_body_builder.config.scalar_initial_state_mass = True
for name in FOUR_BODY_WAVE:
    four_body_builder.dynamics.assign(name, create_relativistic_breit_wigner_with_ff)
four_body_model = four_body_builder.formulate()
four_body_couplings = [
    str(symbol)
    for symbol in four_body_model.parameter_defaults
    if str(symbol).startswith("H")
]
transformer = SympyDataTransformer.from_sympy(
    four_body_model.kinematic_variables, backend="numpy"
)
four_body_function = create_parametrized_function(
    four_body_model.expression.doit(), four_body_model.parameter_defaults, backend="jax"
)
Math(
    aslatex({
        s: v
        for s, v in four_body_model.parameter_defaults.items()
        if str(s).startswith("H")
    })
)

A flat four-body phase-space sample would place almost no events under an 83 keV resonance, so the sample is generated as the decay chain itself: $\overline{B}^0\to D^{*+}\pi^-\pi^0$ with the $D^{*+}$ at its pole mass, followed by an isotropic $D^{*+}\to D^0\pi^+$. That is the narrow-width approximation made explicit, and it is precisely the construction under which the four-body model must reduce to the three-body one when the $D^{*+}$ decay angles are integrated over.

In [ ]:
HELICITY_TAG = {-1: "-1", 0: "0", +1: "+1"}


def generate_phase_space(n_events=500_000, seed=42):
    d_star = phasespace.GenParticle("D*+", MASSES[1]).set_children(
        phasespace.GenParticle("D0", 1.86484),
        phasespace.GenParticle("pi+", MASSES[2]),
    )
    parent = phasespace.GenParticle("B0bar", MASSES[0]).set_children(
        d_star,
        phasespace.GenParticle("pi-", MASSES[2]),
        phasespace.GenParticle("pi0", MASSES[3]),
    )
    weights, particles = parent.generate(n_events=n_events, seed=seed)

    def four_momentum(name):
        p = np.asarray(particles[name])
        return np.stack([p[:, 3], p[:, 0], p[:, 1], p[:, 2]], axis=1)

    momenta = {
        "p0": four_momentum("D0"),
        "p1": four_momentum("pi+"),
        "p2": four_momentum("pi-"),
        "p3": four_momentum("pi0"),
    }
    return np.asarray(weights), momenta


def invariant_mass_squared(*momenta):
    total = sum(momenta)
    return total[:, 0] ** 2 - (total[:, 1:] ** 2).sum(axis=1)


phsp_weights, momenta = generate_phase_space()
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    four_body_data = transformer(momenta)
phsp_sigma1 = invariant_mass_squared(momenta["p2"], momenta["p3"])
phsp_sigma3 = invariant_mass_squared(momenta["p0"], momenta["p1"], momenta["p2"])
phsp_sigma2 = SIGMA_SUM - phsp_sigma1 - phsp_sigma3
three_body_data = {
    "sigma1": phsp_sigma1,
    "sigma2": phsp_sigma2,
    "sigma3": phsp_sigma3,
}
np.testing.assert_allclose(
    invariant_mass_squared(momenta["p0"], momenta["p1"]), MASSES[1] ** 2, rtol=1e-9
)
assert is_inside(phsp_sigma1, phsp_sigma2, phsp_sigma3).all()

## The two models side by side

Each resonance is switched on alone in both models with the same helicity pattern, and the resulting distribution in its own invariant mass is compared. The four-body distribution is the sum of $w\,I_4$ over the generated events, which integrates over the $D^{*+}$ decay angles; the three-body distribution is $w\,I_3$ over the same events. If the narrow-width reduction is faithful, the two must agree in shape.

In [ ]:
def four_body_couplings_for(name, scale=1.0):
    latex = resonances[name].latex
    values = dict.fromkeys(four_body_couplings, 0j)
    values[
        next(c for c in four_body_couplings if c.startswith(R"H_{D^{*}(2010)^{+} \to"))
    ] = 1.0
    if subsystem_ids[name] == 1:
        values[R"H_{\rho(770)^{-} \to \pi^{-}_{0} \pi^{0}_{0}}"] = 1.0
        for helicity, weight in patterns[name].items():
            tag = HELICITY_TAG[helicity]
            key = (
                Rf"H_{{\overline{{B}}^{{0}} \to D^{{*}}(2010)^{{+}}_{{{tag}}} "
                Rf"\rho(770)^{{-}}_{{{tag}}}}}"
            )
            values[key] = scale * weight
        return values
    production = next(
        c
        for c in four_body_couplings
        if c.startswith(Rf"H_{{\overline{{B}}^{{0}} \to {{{latex}}}")
    )
    values[production] = scale
    for helicity, weight in patterns[name].items():
        prefix = Rf"H_{{{latex} \to D^{{*}}(2010)^{{+}}_{{{HELICITY_TAG[helicity]}}}"
        key = next((c for c in four_body_couplings if c.startswith(prefix)), None)
        if key is None:
            assert weight == 0
            continue
        values[key] = weight
    return values


def evaluate_four_body(name):
    four_body_function.update_parameters(four_body_couplings_for(name))
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return np.asarray(four_body_function(four_body_data))


variables = {1: phsp_sigma1, 2: phsp_sigma2, 3: phsp_sigma3}
labels = {
    1: R"$m^2(\pi^-\pi^0)$",
    2: R"$m^2(D^{*+}\pi^0)$",
    3: R"$m^2(D^{*+}\pi^-)$",
}
ranges = {1: (0.3, 1.4), 2: (5.4, 6.6), 3: (5.4, 6.6)}

fig, axes = plt.subplots(2, 3, figsize=(11, 5.4), layout="constrained")
deviations = []
for ax, name in zip(axes.ravel(), DECAY_WAVE, strict=True):
    subsystem = subsystem_ids[name]
    edges = np.linspace(*ranges[subsystem], 51)
    centers = (edges[1:] + edges[:-1]) / 2
    four_body_projection, _ = np.histogram(
        variables[subsystem],
        bins=edges,
        weights=phsp_weights * evaluate_four_body(name),
    )
    three_body_projection, _ = np.histogram(
        variables[subsystem],
        bins=edges,
        weights=phsp_weights * evaluate({name: 1.0}, three_body_data),
    )
    four_body_projection /= four_body_projection.sum()
    three_body_projection /= three_body_projection.sum()
    deviation = (
        np.abs(four_body_projection - three_body_projection)
        / three_body_projection.max()
    )
    deviations.append(deviation)
    ax.step(centers, four_body_projection, where="mid", label="four-body")
    ax.step(
        centers, three_body_projection, where="mid", linestyle="--", label="three-body"
    )
    ax.set(title=f"${resonances[name].latex}$", xlabel=labels[subsystem])
axes[0, 0].legend()
axes[-1, -1].axis("off")
fig.savefig("comparison.svg")
plt.show()
deviations = np.concatenate(deviations)
assert deviations.mean() < 0.03, deviations.mean()
Markdown(
    "Deviation between the normalized projections, relative to each peak: "
    f"{100 * deviations.mean():.1f}% on average and {100 * deviations.max():.1f}% at worst, "
    f"consistent with the Monte Carlo statistics of {len(phsp_weights):,} events."
)

The mass structure of every component survives the narrow-width reduction. Comparing the two models *coherently*, including their interference terms, is a different matter: it requires reconciling the helicity-coupling conventions of AmpForm and AmpForm-DPD, which this report does not attempt.

What the four-body model adds are the two angles of the $D^{*+}$ decay. The polar one, $\cos\theta_{D^{*+}}$, is the angle of the $D^0$ in the $D^{*+}$ rest frame relative to the $D^{*+}$ flight direction in the $\overline{B}^0$ frame; it follows $\left|d^1_{\lambda0}\right|^2$ up to the Monte Carlo noise of the weighted sample, and separates longitudinal from transverse. The azimuthal one is the angle $\varphi$ between the $D^{*+}$ and $\rho^-$ decay planes, and it is the observable that the Dalitz plot does not have: the two transverse sign conventions, which give identical Dalitz densities, give opposite azimuthal correlations.

In [ ]:
def boost_to_rest_frame(momentum, frame):
    mass = np.sqrt(frame[:, 0] ** 2 - (frame[:, 1:] ** 2).sum(axis=1))
    beta = -frame[:, 1:] / frame[:, 0, None]
    beta_squared = (beta**2).sum(axis=1)
    gamma = frame[:, 0] / mass
    dot = (beta * momentum[:, 1:]).sum(axis=1)
    energy = gamma * (momentum[:, 0] + dot)
    factor = (gamma - 1) * dot / beta_squared + gamma * momentum[:, 0]
    return np.column_stack([energy, momentum[:, 1:] + factor[:, None] * beta])


def unit(vector):
    return vector / np.linalg.norm(vector, axis=1)[:, None]


d_star_momentum = momenta["p0"] + momenta["p1"]
d0_direction = unit(boost_to_rest_frame(momenta["p0"], d_star_momentum)[:, 1:])
z_axis = unit(d_star_momentum[:, 1:])
cos_theta = (d0_direction * z_axis).sum(axis=1)
pion_direction = unit(
    boost_to_rest_frame(momenta["p2"], momenta["p2"] + momenta["p3"])[:, 1:]
)
decay_plane_angle = np.arccos(
    np.clip(
        (
            unit(np.cross(z_axis, d0_direction))
            * unit(np.cross(z_axis, pion_direction))
        ).sum(axis=1),
        -1,
        1,
    )
)

polar, azimuthal = {}, {}
saved_pattern = patterns["rho(770)-"]
for label, pattern in (
    ("longitudinal", {-1: 0.0, 0: 1.0, +1: 0.0}),
    ("transverse, equal signs", {-1: 1.0, 0: 0.0, +1: +1.0}),
    ("transverse, opposite signs", {-1: 1.0, 0: 0.0, +1: -1.0}),
    (Rf"$f_L={LONGITUDINAL_FRACTION}$", saved_pattern),
):
    patterns["rho(770)-"] = pattern
    weights = phsp_weights * evaluate_four_body("rho(770)-")
    if "transverse, opposite" not in label:
        polar[label] = weights
    if label.startswith("transverse"):
        azimuthal[label] = weights
patterns["rho(770)-"] = saved_pattern

fig, (left, right) = plt.subplots(1, 2, figsize=(10, 3.6), layout="constrained")
for axis, distributions, variable, edges, xlabel in (
    (left, polar, cos_theta, np.linspace(-1, 1, 21), R"$\cos\theta_{D^{*+}}$"),
    (
        right,
        azimuthal,
        decay_plane_angle,
        np.linspace(0, np.pi, 21),
        R"$\varphi$ [rad]",
    ),
):
    centers = (edges[1:] + edges[:-1]) / 2
    for label, weights in distributions.items():
        projection, _ = np.histogram(variable, bins=edges, weights=weights)
        axis.step(centers, projection / projection.sum(), where="mid", label=label)
    axis.set(xlabel=xlabel, ylabel="normalized yield")
    axis.set_ylim(0, 1.5 * axis.get_ylim()[1])
    axis.legend(fontsize="small", loc="upper center", ncols=2)
left.set_title(R"$D^{*+}$ polar angle")
right.set_title(R"$D^{*+}$-$\rho^-$ decay-plane angle")
fig.savefig("angles.svg")
plt.show()

## Conclusion

The $D^{*+}$ is narrow enough, by a wide margin, for $\overline{B}^0\to D^{*+}\pi^-\pi^0$ to be written as a three-body decay. At $\Gamma/m\approx4\times10^{-5}$ it is the narrowest object in the problem by three orders of magnitude, and every component's mass structure is reproduced by the narrow-width reduction to better than a percent on average.

What the reduction costs is not the mass structure but part of the spin information. Holding the $D^{*+}$ as a stable spin-1 particle already forces one complex coupling per helicity instead of one per chain, and the Dalitz density then depends on those couplings only through combinations that are even in $\lambda$: cross terms between components of opposite $\lambda$-parity vanish pointwise, and the relative phase of the two transverse $\overline{B}^0\to D^{*+}\rho^-$ amplitudes drops out completely. The longitudinal fraction itself survives, so a Dalitz-plot fit can still address the quantity behind the scale factor of $5.2$ on $\mathcal{B}(\overline{B}^0\to D^{*+}\rho^-)$, but the transverse phase needs the decay-plane angle.

Both descriptions are worth having. The three-body model is small enough to build, display and integrate symbolically, and it is the right object for studying the Dalitz structure. The four-body model costs a topology-pinning step in QRules and a decay-chain phase-space generator, and it returns the angles the Dalitz plot integrates away. A full Belle II amplitude analysis of this channel would be a five-dimensional fit, and the second model is the one that describes it.